In [206]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp


In [207]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss


### Read data

In [208]:
regular_season_results = pd.read_csv('../data/MRegularSeasonDetailedResults.csv')
detailed_tourney_results = pd.read_csv('../data/MNCAATourneyDetailedResults.csv')
rankings = pd.read_csv('../data/MMasseyOrdinals.csv')
seeds = pd.read_csv('../data/MNCAATourneySeeds.csv')

# kp_rankings = pd.read_csv('../data/kenpom_pre_tourney_snapshot.csv')

regular_season_results_w = pd.read_csv('../data/WRegularSeasonDetailedResults.csv')
detailed_tourney_results_w = pd.read_csv('../data/WNCAATourneyDetailedResults.csv')

mteams = pd.read_csv('../data/MTeams.csv')
wteams = pd.read_csv('../data/WTeams.csv')

seeds_w = pd.read_csv('../data/WNCAATourneySeeds.csv')

# M538 = pd.read_csv('../data/M538.csv')
# W538 = pd.read_csv('../data/W538.csv')

seed_round = pd.read_csv("../data/MNCAATourneySeedRoundSlots.csv")
seeds = pd.read_csv("../data/MNCAATourneySeeds.csv")

first_round_odds_data = pd.read_csv('../data/sky_data/first_round_odds_ncaam.csv')
first_round_odds_data_w = pd.read_csv('../data/sky_data/first_round_odds_ncaaw.csv')


In [187]:
# [2025, "W01", 1181],
# [2025, "W02", 1104],


In [188]:
# seeds[seeds.Season == 2024]

# RegionW, RegionX, Region Y, Region Z - 
# by our competitions' convention, each of the four regions 
# in the final tournament is assigned a letter of W, X, Y, or Z. 
# Whichever region's name comes first alphabetically, that region will be Region W. 
# And whichever Region plays against Region W in the national semifinals, 
# that will be Region X. 

# For the other two regions, whichever region's name comes first alphabetically, 
# that region will be Region Y, and the other will be Region Z.

# # East -> RegionW
# # Midwest -> RegionX
# # South -> RegionY
# # West -> RegionZ


In [ ]:
torvik_player_data = pd.read_csv("../data/sky_data/torvik_player_data_2008_2025.csv") # this is for sweet 16
torvik_player_data_ncaaw = pd.read_csv("../data/sky_data/ncaaw_torvik_player_data_2021_2025.csv")


In [210]:
mapping = pd.read_csv("../data/sky_data/mappings/kaggle_torvik_mapping.csv")


In [191]:
#aggregated_player_stats = pd.read_csv("../data/sky_data/aggregated_player_stats.csv")

In [211]:
sub_df = pd.read_csv("../data/SampleSubmissionStage2.csv")

### Set Up Data

In [212]:
to_predict_mens, to_predict_womens, regular_season_games, regular_season_games_w = preprocess.full_setup(detailed_tourney_results, regular_season_results,
               detailed_tourney_results_w, regular_season_results_w,
               sub_df, mteams)

/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/submission.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([mens_historical_games, womens_historical_games, sub], axis = 0)


### Add Features

In [213]:
to_predict_womens = feature_engineering.TournamentSeed(tourney_seeds=seeds_w).add(to_predict_womens)
to_predict_womens = feature_engineering.Efficiency(games=regular_season_games_w, away_bonus=0).add(to_predict_womens)
to_predict_womens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_womens)
to_predict_womens = feature_engineering.TeamNames(wteams).add(to_predict_womens)

# to_predict_womens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=W538).add(to_predict_womens)

/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/feature_engineering.py:132: FutureWarning: The provided callable <function mean at 0x1063bc0d0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  final = all_games3.groupby(['Season', 'Team1']).agg(adj_oe=('adj_oe', np.mean), adj_de=('adj_de', np.mean)).reset_index()


In [214]:
to_predict_womens = feature_engineering.FirstRoundOdds(first_round_odds_data_w).add(to_predict_womens)

In [215]:
# to remove later
#to_predict_womens = to_predict_womens[(to_predict_womens.type != "Prediction")].copy()

In [216]:
to_predict_womens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_womens)
to_predict_womens = feature_engineering.AggregatedPlayerStats(torvik_player_data_ncaaw).add(to_predict_womens)

In [198]:
#to_predict_womens.to_csv("../development_notebooks/to_predict_women.csv")
to_predict_womens.to_csv("to_predict_women.csv")

In [217]:
to_predict_mens = feature_engineering.TeamNames(mteams).add(to_predict_mens)
to_predict_mens = feature_engineering.FirstRoundOdds(first_round_odds_data).add(to_predict_mens)
to_predict_mens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_mens)
to_predict_mens = feature_engineering.SeasonStats(regular_season_games).add(to_predict_mens)
# to_predict_mens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=M538).add(to_predict_mens)
to_predict_mens = feature_engineering.PreSeasonAPRankings(rankings_df=rankings).add(to_predict_mens)
to_predict_mens = feature_engineering.TournamentSeed(tourney_seeds=seeds).add(to_predict_mens)
to_predict_mens = feature_engineering.Efficiency(games=regular_season_games, away_bonus=0).add(to_predict_mens)
to_predict_mens = feature_engineering.FinalRanking(rankings_df=rankings, system='WLK').add(to_predict_mens) # switched in 2024 because SAG dissapeared
# to_predict_mens = feature_engineering.Kenpom(kp_snapshot=kp_rankings).add(to_predict_mens)
# this one takes 3 minutes to run
# to_predict_mens = feature_engineering.TeamQuality(games=regular_season_games).add(to_predict_mens)


/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/feature_engineering.py:223: FutureWarning: The provided callable <function mean at 0x1063bc0d0> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  season_statistics = df.groupby(["Season", 'Team1'])[boxscore_cols].agg(np.mean).reset_index()
/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/feature_engineering.py:132: FutureWarning: The provided callable <function mean at 0x1063bc0d0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  final = all_games3.groupby(['Season', 'Team1']).agg(adj_oe=('adj_oe', np.mean), adj_de=('adj_de', np.mean)).reset_index()
/Users/skylerdale/workspace/kaggle-nc

In [218]:
to_predict_mens = feature_engineering.AggregatedPlayerStats(torvik_player_data).add(to_predict_mens)


In [201]:
# Original 

# to_predict_mens = to_predict_mens[(to_predict_mens.type != "Prediction") & 
#                                   (to_predict_mens.final_odds.notnull())
#                                   ]

# New - Not Prediction and (game round != 1 OR final odds is null)

# to_predict_mens = to_predict_mens[
#                                   ( (to_predict_mens.final_odds.notnull()) | (to_predict_mens.GameRound != 1) )
                               
#                                   ]


# to_predict_mens = to_predict_mens[
#                                   ( (to_predict_mens.final_odds.notnull()) | (to_predict_mens.GameRound != 1) )
                               
#                                   ]

### Split Dataset

In [202]:
# first_round_df = to_predict_mens[to_predict_mens.GameRound == 1].copy()
# other_rounds_df = to_predict_mens[to_predict_mens.GameRound > 1].copy()

In [220]:
# first_round_df.to_csv("to_predict_mens_first_round.csv")
# other_rounds_df.to_csv("to_predict_mens_other_rounds.csv")
to_predict_mens.to_csv("to_predict_mens.csv")

### Average odds for first round 


In [204]:
to_predict_mens[(to_predict_mens.t1_Seed == 1)
                & (to_predict_mens.GameRound == 1)].final_odds.mean()

-23.471264367816094

In [205]:
to_predict_mens[(to_predict_mens.t1_Seed == 6)
                & (to_predict_mens.GameRound == 1)].final_odds.mean()

-3.191011235955056